# CAFE: four synchronized camera views -> one ZIP

This notebook downloads **only** a synchronized subset from the public CAFE archive. It selects one place (four cameras), turns each camera's JPEG sequence into one MP4, and creates a ZIP containing the four MP4 files. It does not download the 150 GB archive in full.

The archive supplies frames rather than ready-made videos. CAFE clips are six seconds long and the released frames are sampled at 5 FPS, so the MP4 files are rendered at 5 FPS.

Change `PLACE`, `START_CLIP`, or `TARGET_SOURCE_GIB` only if you want a different synchronized segment. The selected source frames are kept between 15 and 20 GiB; the final MP4 ZIP can be smaller because MP4 re-encodes the JPEG frames.

In [ ]:
import os
import re
import shutil
import struct
import subprocess
import time
import zlib
import zipfile
from collections import defaultdict, namedtuple
from pathlib import Path

import requests

# Public CAFE archive. This URL supports HTTP byte ranges, which lets us fetch only the chosen frames.
ARCHIVE_URL = 'https://drive.usercontent.google.com/download?id=1_JhgdZ6K3ib8xagu6QSL_NKbjIkd-lU6&export=download&confirm=t'

# ---- Selection: change these three values if needed ----
PLACE = 1                 # 1..6. Each place has exactly four synchronized cameras.
START_CLIP = 0            # Start from this six-second clip number.
TARGET_SOURCE_GIB = 16.0  # Must stay in the requested 15..20 GiB range.
# --------------------------------------------------------

assert 1 <= PLACE <= 6, 'PLACE must be from 1 to 6.'
assert 15 <= TARGET_SOURCE_GIB <= 20, 'Use a target between 15 and 20 GiB.'
CAMERA_IDS = list(range((PLACE - 1) * 4 + 1, PLACE * 4 + 1))
FPS = 5
MAX_RANGE_MIB = 64        # A larger value means fewer HTTP requests; 64 MiB is safe for Kaggle RAM.
WORK_ROOT = Path('/kaggle/working/cafe_4view_export')
FINAL_ZIP = Path('/kaggle/working') / f'CAFE_place_{PLACE:02d}_4_synchronized_views.zip'

FFMPEG = shutil.which('ffmpeg')
if not FFMPEG:
    raise RuntimeError('ffmpeg was not found in this Kaggle environment.')

WORK_ROOT.mkdir(parents=True, exist_ok=True)
if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0'})
print(f'Place {PLACE} -> synchronized cameras: {CAMERA_IDS}')
print(f'Output will be: {FINAL_ZIP}')

In [ ]:
# Read only the ZIP directory (about 50 MiB), not the full 150 GB archive.
Member = namedtuple('Member', 'name offset csize usize method')
GIB = 1024 ** 3
FRAME_RE = re.compile(r'.*/Dataset/cafe/(\d+)/(\d+)/images/frames_(\d+)\.jpg$')

def get_range(start, end, attempts=4):
    """Fetch an inclusive byte range with retries."""
    for attempt in range(1, attempts + 1):
        try:
            response = session.get(ARCHIVE_URL, headers={'Range': f'bytes={start}-{end}'}, timeout=(30, 900))
            if response.status_code == 206:
                return response.content
            raise RuntimeError(f'Expected HTTP 206, got HTTP {response.status_code}')
        except Exception as exc:
            if attempt == attempts:
                raise
            print(f'Range retry {attempt}/{attempts}: {exc}')
            time.sleep(2 * attempt)

def read_zip64_extra(extra, needs_usize, needs_csize, needs_offset):
    pos = 0
    while pos + 4 <= len(extra):
        field_id, field_len = struct.unpack_from('<HH', extra, pos)
        data = extra[pos + 4:pos + 4 + field_len]
        pos += 4 + field_len
        if field_id != 0x0001:
            continue
        cursor = 0
        values = []
        for needed in (needs_usize, needs_csize, needs_offset):
            if needed:
                values.append(struct.unpack_from('<Q', data, cursor)[0])
                cursor += 8
            else:
                values.append(None)
        return values
    raise RuntimeError('Required ZIP64 extra field was not found.')

def parse_central_directory(data):
    pos = 0
    while pos + 46 <= len(data) and data[pos:pos + 4] == b'PK\x01\x02':
        header = struct.unpack_from('<4s6H3L5H2L', data, pos)
        method, csize, usize = header[4], header[8], header[9]
        name_len, extra_len, comment_len = header[10], header[11], header[12]
        offset = header[16]
        name_bytes = data[pos + 46:pos + 46 + name_len]
        extra_start = pos + 46 + name_len
        extra = data[extra_start:extra_start + extra_len]
        need_usize, need_csize, need_offset = (usize == 0xFFFFFFFF), (csize == 0xFFFFFFFF), (offset == 0xFFFFFFFF)
        if need_usize or need_csize or need_offset:
            zip64_usize, zip64_csize, zip64_offset = read_zip64_extra(extra, need_usize, need_csize, need_offset)
            usize = zip64_usize if need_usize else usize
            csize = zip64_csize if need_csize else csize
            offset = zip64_offset if need_offset else offset
        name = name_bytes.decode('utf-8', 'replace')
        yield Member(name, offset, csize, usize, method)
        pos += 46 + name_len + extra_len + comment_len

# Discover archive length and its ZIP64 central directory.
probe = session.get(ARCHIVE_URL, headers={'Range': 'bytes=0-0'}, timeout=(30, 120))
probe.raise_for_status()
archive_size = int(probe.headers['Content-Range'].split('/')[-1])
tail_start = max(0, archive_size - 1024 * 1024)
tail = get_range(tail_start, archive_size - 1)
eocd_pos = tail.rfind(b'PK\x05\x06')
locator_pos = tail.rfind(b'PK\x06\x07', 0, eocd_pos)
if eocd_pos < 0 or locator_pos < 0:
    raise RuntimeError('Could not locate the ZIP64 end directory.')
zip64_eocd_offset = struct.unpack_from('<4sLQL', tail, locator_pos)[2]
zip64_eocd = get_range(zip64_eocd_offset, zip64_eocd_offset + 99)
zip64_header = struct.unpack_from('<4sQ2H2L4Q', zip64_eocd, 0)
central_size, central_offset = zip64_header[-2], zip64_header[-1]
central = get_range(central_offset, central_offset + central_size - 1)

# Index JPEG frames for the four cameras of the selected place.
frames = defaultdict(list)
for member in parse_central_directory(central):
    match = FRAME_RE.match(member.name)
    if match and int(match.group(1)) in CAMERA_IDS:
        camera_id, clip_id, frame_id = map(int, match.groups())
        frames[(camera_id, clip_id)].append((frame_id, member))

common_clips = set.intersection(*(set(clip for camera, clip in frames if camera == camera_id) for camera_id in CAMERA_IDS))
target_bytes = int(TARGET_SOURCE_GIB * GIB)
selected_clips, skipped = [], []
selected_bytes = 0
for clip_id in sorted(clip for clip in common_clips if clip >= START_CLIP):
    views = [sorted(frames[(camera_id, clip_id)]) for camera_id in CAMERA_IDS]
    reference_frame_ids = [frame_id for frame_id, _ in views[0]]
    if not all([frame_id for frame_id, _ in view] == reference_frame_ids for view in views[1:]):
        skipped.append(clip_id)
        continue
    clip_bytes = sum(member.csize for view in views for _, member in view)
    if selected_clips and selected_bytes + clip_bytes > target_bytes:
        break
    selected_clips.append(clip_id)
    selected_bytes += clip_bytes

if not selected_clips:
    raise RuntimeError('No synchronized clips fit this selection. Lower START_CLIP or adjust the target.')

camera_members = {camera_id: [] for camera_id in CAMERA_IDS}
for clip_id in selected_clips:
    for camera_id in CAMERA_IDS:
        camera_members[camera_id].extend((clip_id, frame_id, member) for frame_id, member in sorted(frames[(camera_id, clip_id)]))

print(f'Archive size: {archive_size / GIB:.1f} GiB')
print(f'Selected {len(selected_clips)} synchronized clips: {selected_clips[0]}..{selected_clips[-1]}')
print(f'Selected JPEG source: {selected_bytes / GIB:.2f} GiB')
print(f'Per camera: {len(camera_members[CAMERA_IDS[0]])} frames / {len(camera_members[CAMERA_IDS[0]]) / FPS:.1f} seconds')
if skipped:
    print(f'Skipped {len(skipped)} clips with non-matching frame IDs: {skipped[:10]}')

In [ ]:
# Download each camera's frames in larger byte-range batches, make an MP4, then add it to the ZIP.
# Only one camera's JPEG frames are kept on disk at a time.
LOCAL_HEADER = struct.Struct('<4s5H3L2H')
OVERHEAD_BYTES = 128 * 1024
MAX_RANGE_BYTES = MAX_RANGE_MIB * 1024 * 1024

def range_batches(items):
    """Group nearby entries by their position inside the remote ZIP."""
    ordered = sorted(items, key=lambda item: item[0].offset)
    if not ordered:
        return
    batch = [ordered[0]]
    start = ordered[0][0].offset
    end = start + 30 + len(ordered[0][0].name.encode('utf-8')) + OVERHEAD_BYTES + ordered[0][0].csize
    for item in ordered[1:]:
        member = item[0]
        candidate_end = member.offset + 30 + len(member.name.encode('utf-8')) + OVERHEAD_BYTES + member.csize
        if candidate_end - start > MAX_RANGE_BYTES:
            yield start, end, batch
            batch = [item]
            start, end = member.offset, candidate_end
        else:
            batch.append(item)
            end = max(end, candidate_end)
    yield start, end, batch

def extract_camera_frames(camera_id):
    frame_dir = WORK_ROOT / f'camera_{camera_id:02d}_frames'
    frame_dir.mkdir(parents=True, exist_ok=True)
    items = []
    for index, (_, _, member) in enumerate(camera_members[camera_id]):
        items.append((member, frame_dir / f'frame_{index:08d}.jpg'))

    batches = list(range_batches(items))
    for batch_no, (start, end, batch) in enumerate(batches, 1):
        blob = get_range(start, end)
        for member, output_path in batch:
            local_pos = member.offset - start
            signature, _, flags, method, _, _, _, local_csize, _, name_len, extra_len = LOCAL_HEADER.unpack_from(blob, local_pos)
            if signature != b'PK\x03\x04':
                raise RuntimeError(f'Invalid local ZIP header for {member.name}')
            if flags & 0x1:
                raise RuntimeError('Encrypted ZIP entries are not supported.')
            data_start = local_pos + 30 + name_len + extra_len
            # The archive uses ZIP data descriptors, so the local header's compressed size is 0.
            # The central directory contains the real size and is therefore used here.
            compressed = blob[data_start:data_start + member.csize]
            if local_csize not in (0, member.csize) or len(compressed) != member.csize:
                raise RuntimeError(f'Incomplete range for {member.name}')
            if method == 0:
                raw = compressed
            elif method == 8:
                raw = zlib.decompress(compressed, -zlib.MAX_WBITS)
            else:
                raise RuntimeError(f'Unsupported ZIP method {method} for {member.name}')
            if len(raw) != member.usize:
                raise RuntimeError(f'Unexpected decompressed size for {member.name}')
            output_path.write_bytes(raw)
        print(f'Camera {camera_id}: downloaded batch {batch_no}/{len(batches)}')
    return frame_dir

def render_mp4(camera_id, frame_dir):
    manifest = frame_dir / 'frames.txt'
    with manifest.open('w', encoding='utf-8', newline='\n') as handle:
        for image_path in sorted(frame_dir.glob('frame_*.jpg')):
            handle.write(f"file '{image_path.as_posix()}'\n")
    mp4_path = WORK_ROOT / f'CAFE_place_{PLACE:02d}_camera_{camera_id:02d}.mp4'
    command = [
        FFMPEG, '-hide_banner', '-loglevel', 'error', '-y',
        '-f', 'concat', '-safe', '0', '-r', str(FPS), '-i', str(manifest),
        '-an', '-c:v', 'libx264', '-preset', 'medium', '-crf', '17',
        '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(mp4_path),
    ]
    subprocess.run(command, check=True)
    return mp4_path

with zipfile.ZipFile(FINAL_ZIP, mode='w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
    for camera_id in CAMERA_IDS:
        print(f'\n--- Processing camera {camera_id} ---')
        frame_dir = extract_camera_frames(camera_id)
        mp4_path = render_mp4(camera_id, frame_dir)
        archive.write(mp4_path, arcname=mp4_path.name)
        shutil.rmtree(frame_dir)
        mp4_path.unlink()

print(f'\nDone. ZIP size: {FINAL_ZIP.stat().st_size / GIB:.2f} GiB')
print(f'Download it from Kaggle Output: {FINAL_ZIP.name}')
from IPython.display import FileLink, display
display(FileLink(FINAL_ZIP))